# 06 — AI Analysis (Phase 6)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 6 of 8 — AI Executive Summary & Impact Analysis

## 1. Project Overview

This notebook is where Phases 2-5 converge into the object Phase 1's email
pipeline actually needs. Given a publication already in the Phase 3
database, it pulls together its metadata, its Phase 4 extracted text, and
(if one exists) its Phase 5 comparison against a previous version, then
produces a single structured analysis answering exactly the six questions
the project brief specifies:

1. What is this publication about?
2. What changed?
3. Why is it important?
4. When does it become effective?
5. Who is impacted?
6. What action should be considered?

```
Phase 3 DB row (title, category, published_date, urls)
        +
Phase 4 extracted text (data/documents/{id}.txt, if available)
        +
Phase 5 comparison (find_previous_version() + diff_documents(), if a
        previous version exists -- this notebook mirrors that logic
        so it stays self-contained, same convention as every other phase)
        |
        v
generate_ai_analysis()  -- OpenAI if configured, else a tested deterministic
        |                   fallback (same honest pattern as every AI step
        |                   so far in this project)
        v
AIAnalysisResult { summary, key_changes, effective_date, impact_level,
                    impacted_departments, recommended_actions }
        |
        v
build_dgca_update()  -->  DGCAUpdate  (Phase 1's exact schema, mirrored here
                            to confirm this notebook's output is actually
                            ready to feed Phase 1 -- without duplicating
                            Phase 1's email-rendering code, which stays
                            Phase 1's responsibility)
```

**Scope of this phase:** producing a validated `DGCAUpdate`-ready analysis
object and saving it to disk. Actually calling Phase 1's
`send_dgca_notification()` with it -- i.e. wiring Phases 2 through 6 into one
live run -- is deliberately left to Phase 7 (agentic orchestration) or a
lightweight glue script, consistent with every "not yet wired together" note
left in Phases 3-5.


## 2. Imports

In [ ]:
import difflib
import json
import logging
import os
import re
from psycopg import Connection
from psycopg.rows import DictRow
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Optional, Tuple

from dotenv import load_dotenv
from pydantic import BaseModel, Field

try:
    from openai import OpenAI
    _OPENAI_SDK_AVAILABLE = True
except ImportError:
    _OPENAI_SDK_AVAILABLE = False

print("Imports OK")


## 3. Configuration

Same `OPENAI_API_KEY`/`LLM_MODEL` as Phases 1 and 5. Runs completely without
either configured -- the deterministic fallback is what gets tested below,
same honest pattern as every prior AI step in this project.


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ENV_PATH = PROJECT_ROOT / "config" / ".env"
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed" / "analysis"
LOG_DIR = PROJECT_ROOT / "logs"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(dotenv_path=ENV_PATH)

_PLACEHOLDER_PATTERNS = ("your-", "00000000-0000-0000-0000-000000000000", "sk-your-")


def _is_placeholder(value: Optional[str]) -> bool:
    if not value:
        return True
    return any(value.startswith(p) for p in _PLACEHOLDER_PATTERNS)


@dataclass
class AnalysisConfig:
    openai_api_key: Optional[str] = None
    llm_model: str = "gpt-4o-mini"

    def openai_configured(self) -> bool:
        return not _is_placeholder(self.openai_api_key)


def load_config() -> AnalysisConfig:
    return AnalysisConfig(
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        llm_model=os.getenv("LLM_MODEL", "gpt-4o-mini"),
    )


config = load_config()
print(f"OPENAI_API_KEY configured: {config.openai_configured()}")
print(f"LLM_MODEL: {config.llm_model}")

from pipeline.storage import get_connection


## 4. Logging

In [ ]:
LOG_FILE = LOG_DIR / f"ai_analysis_{datetime.now(timezone.utc).strftime('%Y%m%d')}.log"

logger = logging.getLogger("dgca_ai_analysis")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.propagate = False

logger.info("AI analysis logger initialized.")
logger.info(f"Log file: {LOG_FILE}")


## 5. Comparison Helpers (mirrors Phase 5)

Self-contained, same convention as every other notebook in this project --
this reproduces Phase 5's whitespace-normalized diff engine and
previous-version matching exactly, since Phase 6 needs to compute "what
changed" for any publication that turns out to have a previous version.
**Keep this in sync with `05_Old_New_Comparison.ipynb` if that notebook's
diff logic changes.**


In [ ]:
def normalize_for_diff(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.;:])\s+")


def split_into_units(text: str, min_len: int = 8) -> List[str]:
    normalized = normalize_for_diff(text)
    raw_units = _SENTENCE_SPLIT_RE.split(normalized)
    return [u.strip() for u in raw_units if len(u.strip()) >= min_len]


@dataclass
class DiffResult:
    added: List[str] = field(default_factory=list)
    removed: List[str] = field(default_factory=list)
    modified: List[Tuple[str, str]] = field(default_factory=list)
    unchanged_count: int = 0


def diff_documents(old_text: str, new_text: str) -> DiffResult:
    old_units = split_into_units(old_text)
    new_units = split_into_units(new_text)
    matcher = difflib.SequenceMatcher(None, old_units, new_units, autojunk=False)
    result = DiffResult()
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            result.unchanged_count += (i2 - i1)
        elif tag == "insert":
            result.added.extend(new_units[j1:j2])
        elif tag == "delete":
            result.removed.extend(old_units[i1:i2])
        elif tag == "replace":
            old_chunk, new_chunk = old_units[i1:i2], new_units[j1:j2]
            paired = min(len(old_chunk), len(new_chunk))
            for k in range(paired):
                result.modified.append((old_chunk[k], new_chunk[k]))
            result.removed.extend(old_chunk[paired:])
            result.added.extend(new_chunk[paired:])
    return result


_REVISION_SUFFIX_RE = re.compile(r"\b(revision|rev\.?|amendment|issue)\s*\d+\b", re.IGNORECASE)


def normalize_title_for_matching(title: str) -> str:
    stripped = _REVISION_SUFFIX_RE.sub("", title)
    return re.sub(r"\s+", " ", stripped).strip().lower()


def titles_likely_same_document(title_a: str, title_b: str, threshold: float = 0.85) -> bool:
    norm_a, norm_b = normalize_title_for_matching(title_a), normalize_title_for_matching(title_b)
    if not norm_a or not norm_b:
        return False
    return difflib.SequenceMatcher(None, norm_a, norm_b).ratio() >= threshold


def find_previous_version(conn: Connection, publication_id: str) -> Optional[DictRow]:
    current = conn.execute("SELECT * FROM publications WHERE publication_id = %s", (publication_id,)).fetchone()
    if current is None:
        return None
    candidates = conn.execute(
        "SELECT * FROM publications WHERE category = %s AND publication_id != %s ORDER BY first_seen_at",
        (current["category"], publication_id),
    ).fetchall()
    for candidate in candidates:
        if titles_likely_same_document(current["title"], candidate["title"]):
            return candidate
    return None


class ComparisonSummary(BaseModel):
    overall_change_level: str = "NONE"
    total_changes: int = 0
    key_changes: List[str] = Field(default_factory=list)


def build_comparison_summary(diff: DiffResult) -> ComparisonSummary:
    total = len(diff.added) + len(diff.removed) + len(diff.modified)
    if total == 0:
        level = "NONE"
    elif total <= 2:
        level = "LOW"
    elif total <= 6:
        level = "MEDIUM"
    else:
        level = "HIGH"

    key_changes: List[str] = []
    for item in diff.added[:5]:
        key_changes.append(f"Added: {item}")
    for item in diff.removed[:5]:
        key_changes.append(f"Removed: {item}")
    for old, new in diff.modified[:5]:
        key_changes.append(f"Modified: \'{old}\' -> \'{new}\'")

    return ComparisonSummary(overall_change_level=level, total_changes=total, key_changes=key_changes)


print("Comparison helpers ready (mirrors Phase 5).")


## 6. Text Utilities

`extract_effective_date_hint()` is a best-effort heuristic for the
deterministic fallback (real DGCA documents often state "EFFECTIVE:
FORTHWITH" or a specific date near that word) -- the real LLM path does
this far more reliably from full context; this is only the no-API-key
safety net.


In [ ]:
_EFFECTIVE_DATE_RE = re.compile(r"effective[:\s]+([^.\n]{2,40})", re.IGNORECASE)


def extract_effective_date_hint(normalized_text: str) -> str:
    match = _EFFECTIVE_DATE_RE.search(normalized_text)
    if match:
        candidate = match.group(1).strip().rstrip(".,;: ")
        if candidate:
            return candidate[:60]
    return "Not specified in the publication."


print("Text utilities ready.")


## 7. Data Models

`AIAnalysisResult` maps directly onto Phase 1's `DGCAUpdate` fields.
`DGCAUpdate` here is a **mirror of Phase 1's exact schema** -- reproduced
locally (not imported, notebooks stay independent) purely so this notebook
can prove its own output actually validates against what Phase 1 expects,
without duplicating Phase 1's email-generation logic.


In [ ]:
class AIAnalysisResult(BaseModel):
    summary: str
    key_changes: List[str] = Field(default_factory=list)
    effective_date: str = "Not specified in the publication."
    impact_level: str = "MEDIUM"  # LOW | MEDIUM | HIGH | CRITICAL
    impacted_departments: List[str] = Field(default_factory=list)
    recommended_actions: List[str] = Field(default_factory=list)
    analysis_mode: str = "fallback"  # "ai" | "fallback" | "degraded"


class DGCAUpdate(BaseModel):
    """Mirrors notebooks/01_DGCA_Email_Pipeline.ipynb\'s DGCAUpdate exactly."""

    title: str
    category: str
    published_date: str
    summary: str
    key_changes: List[str] = Field(default_factory=list)
    effective_date: str = "Not specified in the publication."
    impact_level: str = "MEDIUM"
    impacted_departments: List[str] = Field(default_factory=list)
    recommended_actions: List[str] = Field(default_factory=list)
    dgca_url: Optional[str] = None
    pdf_url: Optional[str] = None


print("Data models ready.")


## 8. AI Analysis Agent

Same honest pattern as every AI step so far: OpenAI when configured, a
deterministic rule-based builder otherwise -- logged clearly as a fallback,
never silently pretended to be AI-generated. Three distinct situations are
handled explicitly rather than lumped together: a previous version *with*
detected changes, a previous version with *no* substantive changes, and no
previous version at all (first-time publication) -- plus a fully degraded
path when Phase 4 couldn't extract any text at all (e.g. `needs_ocr`).


In [ ]:
MAX_TEXT_CHARS_FOR_PROMPT = 12000


def _fallback_analysis(row: DictRow, extracted_text: Optional[str], comparison: Optional[ComparisonSummary]) -> AIAnalysisResult:
    title, category = row["title"], row["category"]
    has_text = bool(extracted_text and extracted_text.strip())

    if not has_text:
        return AIAnalysisResult(
            summary=(
                f"{title} ({category}). Automated text extraction was not available for this "
                "document (see Phase 4 processing status); manual review of the PDF is required "
                "before an executive summary can be generated."
            ),
            key_changes=[],
            effective_date="Not specified in the publication.",
            impact_level="MEDIUM",
            impacted_departments=[],
            recommended_actions=["Manually review the source PDF -- automated text extraction was unavailable."],
            analysis_mode="degraded",
        )

    normalized = normalize_for_diff(extracted_text)
    preview = normalized[:400]
    effective_date = extract_effective_date_hint(normalized)

    if comparison is not None and comparison.total_changes > 0:
        what_changed = (
            f"{comparison.total_changes} change(s) identified relative to the previous version "
            f"(change level: {comparison.overall_change_level})."
        )
        key_changes = comparison.key_changes
        impact_level = comparison.overall_change_level if comparison.overall_change_level in ("LOW", "MEDIUM", "HIGH") else "MEDIUM"
        recommended_actions = ["Review the specific changes identified above against current procedures."]
    elif comparison is not None:
        what_changed = "No substantive changes detected relative to the previous version."
        key_changes = []
        impact_level = "LOW"
        recommended_actions = []
    else:
        what_changed = "No previous version is on record; this appears to be a first-time publication."
        key_changes = []
        impact_level = "MEDIUM"
        recommended_actions = ["Review full document content for detailed compliance requirements."]

    summary = f"{title} ({category}). {what_changed} Content preview: {preview}..."

    return AIAnalysisResult(
        summary=summary,
        key_changes=key_changes,
        effective_date=effective_date,
        impact_level=impact_level,
        impacted_departments=[],
        recommended_actions=recommended_actions,
        analysis_mode="fallback",
    )


def generate_ai_analysis(
    row: DictRow,
    extracted_text: Optional[str],
    comparison: Optional[ComparisonSummary],
    config: AnalysisConfig,
) -> AIAnalysisResult:
    has_text = bool(extracted_text and extracted_text.strip())

    if not has_text:
        logger.warning(f"[{row['publication_id']}] No extracted text available -- using degraded fallback.")
        return _fallback_analysis(row, extracted_text, comparison)

    if not (_OPENAI_SDK_AVAILABLE and config.openai_configured()):
        logger.warning(f"[{row['publication_id']}] OPENAI_API_KEY not configured -- using deterministic fallback analysis.")
        return _fallback_analysis(row, extracted_text, comparison)

    normalized = normalize_for_diff(extracted_text)[:MAX_TEXT_CHARS_FOR_PROMPT]
    comparison_block = (
        comparison.model_dump_json(indent=2) if comparison is not None
        else "No previous version on record -- this is a first-time publication."
    )
    prompt = (
        "You are a regulatory intelligence analyst for an airline. Read the DGCA publication\n"
        "metadata, its extracted text, and (if present) a summary of changes versus its previous\n"
        "version. Answer six questions and return them as a single JSON object with EXACTLY these\n"
        "keys: summary (string -- combine what this publication is about and why it matters),\n"
        "key_changes (list of strings -- what changed), effective_date (string -- when it takes\n"
        "effect), impact_level (one of LOW|MEDIUM|HIGH|CRITICAL), impacted_departments (list of\n"
        "strings -- who is impacted), recommended_actions (list of strings -- what action to\n"
        "consider).\n\n"
        "Rules:\n"
        "- Never invent requirements not present in the text or comparison.\n"
        "- If information is not available, use exactly: \"Not specified in the publication.\"\n"
        "- Prioritize substantive regulatory/operational content over formatting.\n\n"
        f"Title: {row['title']}\n"
        f"Category: {row['category']}\n"
        f"Published: {row['published_date']}\n\n"
        f"COMPARISON SUMMARY:\n{comparison_block}\n\n"
        f"DOCUMENT TEXT (may be truncated):\n{normalized}\n"
    )
    try:
        client = OpenAI(api_key=config.openai_api_key)
        response = client.chat.completions.create(
            model=config.llm_model,
            messages=[
                {"role": "system", "content": "You produce accurate, structured JSON regulatory analyses for an airline. Output valid JSON only."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
            response_format={"type": "json_object"},
        )
        parsed = json.loads(response.choices[0].message.content)
        parsed["analysis_mode"] = "ai"
        result = AIAnalysisResult(**parsed)
        logger.info(f"AI analysis generated via OpenAI model \'{config.llm_model}\'.")
        return result
    except Exception as exc:
        logger.error(f"AI analysis generation failed ({exc.__class__.__name__}). Falling back to deterministic analysis.")
        return _fallback_analysis(row, extracted_text, comparison)


print("AI analysis (+ fallback) ready.")


## 9. Building the Final `DGCAUpdate`

In [ ]:
def build_dgca_update(row: DictRow, analysis: AIAnalysisResult) -> DGCAUpdate:
    return DGCAUpdate(
        title=row["title"],
        category=row["category"],
        published_date=row["published_date"] or "Not specified in the publication.",
        summary=analysis.summary,
        key_changes=analysis.key_changes,
        effective_date=analysis.effective_date,
        impact_level=analysis.impact_level,
        impacted_departments=analysis.impacted_departments,
        recommended_actions=analysis.recommended_actions,
        dgca_url=row["publication_url"],
        pdf_url=row["pdf_url"],
    )


print("build_dgca_update() ready.")


## 10. Orchestrator — `analyze_publication()`

Loads everything for a given `publication_id` from disk/DB, computes the
comparison if a previous version exists, and returns both the raw
`AIAnalysisResult` and the ready-to-use `DGCAUpdate`. Raises a clear
`ValueError` for an unknown `publication_id` -- a caller passing a
nonexistent id is a caller bug, not an environmental failure, so this is
intentionally not swallowed the way network/parsing failures are elsewhere
in this project.


In [ ]:
def analyze_publication(conn: Connection, publication_id: str, config: AnalysisConfig) -> Tuple[AIAnalysisResult, DGCAUpdate]:
    row = conn.execute("SELECT * FROM publications WHERE publication_id = %s", (publication_id,)).fetchone()
    if row is None:
        raise ValueError(f"Publication \'{publication_id}\' not found in database.")

    text_path = DOCUMENTS_DIR / f"{publication_id}.txt"
    extracted_text = text_path.read_text(encoding="utf-8") if text_path.exists() else None

    comparison: Optional[ComparisonSummary] = None
    previous_row = find_previous_version(conn, publication_id)
    if previous_row is not None:
        old_text_path = DOCUMENTS_DIR / f"{previous_row['publication_id']}.txt"
        if old_text_path.exists() and extracted_text:
            diff = diff_documents(old_text_path.read_text(encoding="utf-8"), extracted_text)
            comparison = build_comparison_summary(diff)
            logger.info(
                f"[{publication_id}] Previous version found: {previous_row['publication_id']} -- "
                f"{comparison.total_changes} change(s)."
            )
        else:
            logger.warning(
                f"[{publication_id}] Previous version {previous_row['publication_id']} found, "
                "but text unavailable for one or both documents -- skipping comparison."
            )
    else:
        logger.info(f"[{publication_id}] No previous version found.")

    analysis = generate_ai_analysis(row, extracted_text, comparison, config)
    update = build_dgca_update(row, analysis)
    return analysis, update


print("analyze_publication() ready.")


## 11. Test Data — constructed scenarios for `generate_ai_analysis()`

Real DGCA data has no genuine revision pair (established in Phase 5), so the
first three tests exercise `generate_ai_analysis()` directly with
constructed inputs -- precise and deterministic, same approach as testing
Phase 5's diff engine before layering the DB lookup on top. A real
publication row shape is reused so the tests stay realistic.


In [ ]:
_test_row = {
    "publication_id": "test-pub",
    "title": "CAR Section 1 - Revision 2",
    "category": "Regulation",
    "published_date": "2026-08-17",
    "publication_url": "https://www.dgca.gov.in/digigov-portal/",
    "pdf_url": "https://example.com/dummy.pdf",
}

_real_text_path = DOCUMENTS_DIR / "regulation-yv5UwjfEILZBCtSQvOHdA.txt"
_real_text = _real_text_path.read_text(encoding="utf-8")
print(f"Loaded real extracted text: {len(_real_text)} characters from {_real_text_path.name}")


## 12. Test 1 — Previous version found, changes detected

In [ ]:
_diff_with_changes = diff_documents(_real_text, _real_text + " Operators must submit updated compliance documentation within 30 days.")
_comparison_with_changes = build_comparison_summary(_diff_with_changes)

analysis_1 = generate_ai_analysis(_test_row, _real_text, _comparison_with_changes, config)
print(analysis_1.model_dump_json(indent=2))

assert analysis_1.analysis_mode == "fallback"
assert len(analysis_1.key_changes) > 0
assert analysis_1.impact_level in ("LOW", "MEDIUM", "HIGH")
print("Test 1 passed: comparison-driven key_changes and impact_level populated.")


## 13. Test 2 — No previous version (first-time publication)

In [ ]:
analysis_2 = generate_ai_analysis(_test_row, _real_text, None, config)
print(analysis_2.model_dump_json(indent=2))

assert analysis_2.analysis_mode == "fallback"
assert analysis_2.key_changes == []
assert "first-time" in analysis_2.summary.lower() or "no previous version" in analysis_2.summary.lower()
print("Test 2 passed: first-time publication correctly noted, no phantom key_changes.")


## 14. Test 3 — No extracted text available (degraded, e.g. `needs_ocr`)

In [ ]:
analysis_3 = generate_ai_analysis(_test_row, None, None, config)
print(analysis_3.model_dump_json(indent=2))

assert analysis_3.analysis_mode == "degraded"
assert "manual" in analysis_3.recommended_actions[0].lower()
print("Test 3 passed: missing text handled gracefully, not a crash, not a fabricated summary.")


## 15. Test 4 — Real database integration (real publication)

Exercises the full `analyze_publication()` orchestrator against the actual
Phase 3 database and Phase 4 extracted text -- no construction, no mocking.


In [ ]:
real_conn = get_connection()

analysis_4, update_4 = analyze_publication(real_conn, "regulation-yv5UwjfEILZBCtSQvOHdA", config)
print(update_4.model_dump_json(indent=2))

assert isinstance(update_4, DGCAUpdate)
assert update_4.title == "DGCA Example Regulation" or len(update_4.title) > 0
assert analysis_4.analysis_mode in ("fallback", "ai")
print("Test 4 passed: real publication analyzed end-to-end, DGCAUpdate validates.")


## 16. Test 5 — Real database integration (publication with no extracted text)

In [ ]:
analysis_5, update_5 = analyze_publication(real_conn, "tender-eVzkArZYTgSmhfx9huNw", config)
print(update_5.model_dump_json(indent=2))

assert analysis_5.analysis_mode == "degraded"
assert isinstance(update_5, DGCAUpdate)
print("Test 5 passed: scanned/no-text publication degrades gracefully, still produces a valid DGCAUpdate.")


## 17. Test 6 — Full orchestrator with a synthetic revision pair

Seeds an in-memory database (same technique as Phase 5's Test 5b) plus two
temporary `.txt` files, so `analyze_publication()`'s previous-version lookup
and comparison logic both run for real, end to end, not just their
individual pieces.


In [ ]:
_PUBLICATIONS_SCHEMA = """
CREATE TEMP TABLE publications (
    publication_id   TEXT PRIMARY KEY,
    title             TEXT NOT NULL,
    category          TEXT NOT NULL,
    published_date    TEXT,
    publication_url   TEXT,
    pdf_url           TEXT,
    content_hash      TEXT NOT NULL,
    document_hash     TEXT,
    first_seen_at     TEXT NOT NULL,
    processed_at      TEXT,
    email_sent        INTEGER NOT NULL DEFAULT 0,
    email_sent_at     TEXT,
    status            TEXT NOT NULL DEFAULT \'new\'
);
"""

_old_id, _new_id = "test-car-x-rev1", "test-car-x-rev2"
_old_text_path = DOCUMENTS_DIR / f"{_old_id}.txt"
_new_text_path = DOCUMENTS_DIR / f"{_new_id}.txt"
_old_text_path.write_text(_real_text, encoding="utf-8")
_new_text_path.write_text(_real_text + " All operators must comply with this revision within 60 days.", encoding="utf-8")

test_conn = get_connection()
test_conn.execute(_PUBLICATIONS_SCHEMA)
test_conn.cursor().executemany(
    "INSERT INTO publications (publication_id, title, category, published_date, publication_url, pdf_url, content_hash, first_seen_at) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
    [
        (_old_id, "CAR Section X - Revision 1", "Regulation", "2026-01-01",
         "https://www.dgca.gov.in/digigov-portal/", "https://example.com/old.pdf", "hashold", "2026-01-01T00:00:00+00:00"),
        (_new_id, "CAR Section X - Revision 2", "Regulation", "2026-06-01",
         "https://www.dgca.gov.in/digigov-portal/", "https://example.com/new.pdf", "hashnew", "2026-06-01T00:00:00+00:00"),
    ],
)
test_conn.commit()

try:
    analysis_6, update_6 = analyze_publication(test_conn, _new_id, config)
    print(update_6.model_dump_json(indent=2))
    assert len(analysis_6.key_changes) > 0, "Expected the synthetic revision to produce detected key_changes."
    print("Test 6 passed: full orchestrator correctly found the previous version and derived key_changes from it.")
finally:
    test_conn.close()
    _old_text_path.unlink(missing_ok=True)
    _new_text_path.unlink(missing_ok=True)


## 18. Test 7 — Unknown `publication_id` (error handling)

In [ ]:
try:
    analyze_publication(real_conn, "this-id-does-not-exist", config)
    raise AssertionError("Expected ValueError for an unknown publication_id.")
except ValueError as exc:
    print(f"Test 7 passed: unknown publication_id raised a clear error: {exc}")


## 19. Save Results

In [ ]:
output_path = ANALYSIS_DIR / "regulation-yv5UwjfEILZBCtSQvOHdA.json"
output_path.write_text(update_4.model_dump_json(indent=2), encoding="utf-8")
print(f"Saved DGCAUpdate-ready analysis to: {output_path}")

real_conn.close()


## 20. Results

In [ ]:
print("=" * 70)
print("AI ANALYSIS TEST RESULTS")
print("=" * 70)
print("Test 1 (previous version, changes detected)     : PASSED")
print("Test 2 (no previous version)                     : PASSED")
print("Test 3 (degraded, no extracted text)             : PASSED")
print("Test 4 (real DB, real publication)                : PASSED")
print("Test 5 (real DB, no extracted text)               : PASSED")
print("Test 6 (real orchestrator, synthetic revision)    : PASSED")
print("Test 7 (unknown publication_id)                   : PASSED")
print(f"AI analysis mode                                  : {'OpenAI (' + config.llm_model + ')' if config.openai_configured() else 'deterministic fallback (no OPENAI_API_KEY)'}")
print("=" * 70)


## 21. Next Steps

**Phase 6 status: AI analysis built and tested** against constructed
scenarios (with/without a previous version, with/without extracted text)
and real Phase 3/4 data, including a full end-to-end run through a
synthetic revision pair that exercises previous-version lookup, diffing,
and analysis generation together.

### Known limitation

- The real OpenAI path is implemented and documented but not exercised for
  real here -- no `OPENAI_API_KEY` configured, same situation as every
  other AI step in this project. All tests validate the deterministic
  fallback, which produces the identical `AIAnalysisResult` schema.

### Not started yet (by design)

- Phase 7: Agentic workflow orchestration -- this is what would actually
  chain Phase 2 (scrape) -> Phase 3 (store) -> Phase 4 (process) -> Phase 5
  (compare) -> Phase 6 (analyze, this notebook) -> Phase 1 (email) into one
  live run, rather than each notebook being exercised independently as they
  have been so far
- Phase 8: Streamlit dashboard

Waiting for confirmation before starting Phase 7.
